# HomeTown AR: product photo to 3D model with Hunyuan3D-2 (sharper than TripoSR)
Runtime > Change runtime type > **T4 GPU**. Run cells 1, 2, 3, 4 in order. Cell 2b (textures) is optional and needs more GPU memory.
Upload `products.json` from `node scripts/colab-batch.mjs export --next 20`. Download `models.zip` and run
`node scripts/colab-batch.mjs import models.zip` then `node scripts/bundle-models.mjs`.
Without the texture step each model gets a clean shape painted in the product's main colour; with it, the photo's real surface detail.

In [ ]:
!nvidia-smi -L
!rm -rf /content/Hunyuan3D-2
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2 /content/Hunyuan3D-2
%cd /content/Hunyuan3D-2
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install rembg onnxruntime
# after this finishes: Runtime > Restart session, then run cell 3 (not this one again)

In [ ]:
# OPTIONAL: photo textures (needs a GPU with enough memory; skip if it errors, the models still work)
%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!python setup.py install
%cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
!python setup.py install
%cd /content

In [ ]:
from google.colab import files
files.upload()                    # choose products.json
import json, io, os, sys, zipfile, requests, torch, trimesh, numpy as np
from PIL import Image
sys.path.insert(0, '/content/Hunyuan3D-2')
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
products = json.load(open('products.json'))
print(len(products), 'products')
assert torch.cuda.is_available(), 'Turn on a GPU: Runtime > Change runtime type > T4 GPU'
shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini', variant='fp16')
rembg_ = BackgroundRemover()
paint = None
try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
    print('textures: on')
except Exception as e:
    print('textures: off (', str(e)[:120], ') -> clean shape in the product colour')
os.makedirs('out', exist_ok=True)
def tint(mesh, hexcol):
    c = tuple(int(hexcol.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)) + (255,)
    mesh.visual = trimesh.visual.ColorVisuals(mesh=mesh, vertex_colors=np.tile(np.array(c, dtype=np.uint8), (len(mesh.vertices), 1)))
    return mesh
for p in products:
    try:
        img = Image.open(io.BytesIO(requests.get(p['photo'], timeout=60).content)).convert('RGBA')
        img = rembg_(img.convert('RGB'))
        mesh = shape(image=img, num_inference_steps=50, octree_resolution=380, num_chunks=20000, generator=torch.manual_seed(12355))[0]
        done = False
        if paint is not None:
            try:
                mesh = paint(mesh, image=img); done = True
            except Exception as e:
                print('  texture failed for', p['id'], str(e)[:80]); torch.cuda.empty_cache()
        if not done: mesh = tint(mesh, p.get('dominantHex', '#b8a58f'))
        mesh.export(f"out/{p['id']}.glb")
        print('made', p['id'], '(textured)' if done else '(plain colour)')
        torch.cuda.empty_cache()
    except Exception as e:
        print('FAILED', p['id'], str(e)[:200])

In [ ]:
with zipfile.ZipFile('models.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('out')): z.write(f'out/{f}', f)
files.download('models.zip')